# ARIMA and SES Benchmark for Bitcoin Daily Returns

This notebook builds classical benchmark models for the same target used by the LSTM and LSTM-SGARCH notebooks: **next-day Bitcoin daily log return**.

Target definition:

```text
r_t = log(P_t / P_{t-1})
y_t = r_{t+1}
```

The benchmark is intentionally simple and reproducible:

- construct the daily close-to-close log-return target from the raw 5-minute Coinbase BTC candles,
- use the same last-30-days validation and last-30-days test convention as the LSTM notebooks,
- tune ARIMA order on the training segment only,
- evaluate naive, rolling-mean, SES, and ARIMA forecasts with walk-forward updates.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import warnings

Path("/private/tmp/matplotlib-cache").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-cache")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.6f}".format


In [ ]:
RAW_5MIN_PATH = Path("data/BTC/BTC_USD_coinbase_spot_5min.csv")
OUT_DIR = Path("artifacts/arima_daily_return")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP_COL = "timestamp_utc"
TARGET_COL = "target_next_daily_log_return"

VAL_PERIOD_DAYS = 30
TEST_PERIOD_DAYS = 30
TRAIN_WINDOW_DAYS = 30
EPS = 1e-12


## Build the Daily Return Target

The target is built directly from daily closes derived from the raw 5-minute file. This avoids using the volatility target and keeps ARIMA aligned with the project target variable.


In [ ]:
raw_5min = pd.read_csv(RAW_5MIN_PATH)
raw_5min[TIMESTAMP_COL] = pd.to_datetime(raw_5min[TIMESTAMP_COL], utc=True)
raw_5min = raw_5min.sort_values(TIMESTAMP_COL).reset_index(drop=True)

raw_5min["log_close_5m"] = np.log(raw_5min["close"])
raw_5min["log_return_5m"] = raw_5min["log_close_5m"].diff()

daily = raw_5min.set_index(TIMESTAMP_COL).resample("1D").agg(close=("close", "last"))
daily_intraday_vol = raw_5min.set_index(TIMESTAMP_COL)["log_return_5m"].resample("1D").std().rename("daily_volatility")

daily = daily.join(daily_intraday_vol)
daily["daily_log_return"] = np.log(daily["close"] / daily["close"].shift(1))
daily["volatility_7d"] = daily["daily_volatility"].rolling(7).mean()
daily[TARGET_COL] = daily["daily_log_return"].shift(-1)

return_data = daily.reset_index()[[TIMESTAMP_COL, "daily_log_return", "daily_volatility", "volatility_7d", TARGET_COL]].dropna()
return_data = return_data.sort_values(TIMESTAMP_COL).reset_index(drop=True)

max_time = return_data[TIMESTAMP_COL].max()
test_start = max_time - pd.Timedelta(days=TEST_PERIOD_DAYS)
val_start = test_start - pd.Timedelta(days=VAL_PERIOD_DAYS)

return_data["split"] = np.select(
    [
        return_data[TIMESTAMP_COL] < val_start,
        (return_data[TIMESTAMP_COL] >= val_start) & (return_data[TIMESTAMP_COL] < test_start),
        return_data[TIMESTAMP_COL] >= test_start,
    ],
    ["train", "validation", "test"],
    default="unused",
)

split_summary = return_data.groupby("split").agg(
    rows=(TARGET_COL, "size"),
    start=(TIMESTAMP_COL, "min"),
    end=(TIMESTAMP_COL, "max"),
    target_mean=(TARGET_COL, "mean"),
    target_std=(TARGET_COL, "std"),
)

print("Prepared daily rows:", len(return_data))
print("Target:", "target_next_daily_log_return = daily_log_return.shift(-1)")
print("Validation starts:", val_start)
print("Test starts:", test_start)
display(split_summary)
return_data.head()


## Train, Validation, and Test Series

In [ ]:
train = return_data.loc[return_data["split"] == "train", TARGET_COL].astype(float)
validation = return_data.loc[return_data["split"] == "validation", TARGET_COL].astype(float)
test = return_data.loc[return_data["split"] == "test", TARGET_COL].astype(float)

train.index = return_data.loc[return_data["split"] == "train", TIMESTAMP_COL]
validation.index = return_data.loc[return_data["split"] == "validation", TIMESTAMP_COL]
test.index = return_data.loc[return_data["split"] == "test", TIMESTAMP_COL]

print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print(f"Test rows: {len(test):,}")
print(f"Train period: {train.index.min().date()} to {train.index.max().date()}")
print(f"Validation period: {validation.index.min().date()} to {validation.index.max().date()}")
print(f"Test period: {test.index.min().date()} to {test.index.max().date()}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(train.index, train, label="Train", linewidth=0.8)
axes[0].plot(validation.index, validation, label="Validation", linewidth=0.9)
axes[0].plot(test.index, test, label="Test", linewidth=0.9)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Daily log-return target split")
axes[0].set_ylabel("Log return")
axes[0].legend()

sns.histplot(train, bins=80, kde=True, ax=axes[1], color="tab:blue")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Training target distribution")
axes[1].set_xlabel("Next-day daily log return")

plt.tight_layout()
plt.show()


## Simple Walk-Forward Benchmarks

These are baseline return forecasts, not volatility forecasts:

- `naive`: tomorrow's return equals today's observed target value,
- `rolling_7`, `rolling_14`, `rolling_30`: mean of recent observed target values,
- `ses`: Simple Exponential Smoothing level forecast.

For validation and test, the true value is appended to the history after each one-step forecast.


In [ ]:
def benchmark_forecasts(history: list[float], actuals: pd.Series) -> pd.DataFrame:
    rows = []
    history = list(history)

    for date, actual in actuals.items():
        rows.append({
            TIMESTAMP_COL: date,
            "actual": float(actual),
            "naive": float(history[-1]),
            "rolling_7": float(np.mean(history[-7:])),
            "rolling_14": float(np.mean(history[-14:])),
            "rolling_30": float(np.mean(history[-30:])),
        })
        history.append(float(actual))

    return pd.DataFrame(rows).set_index(TIMESTAMP_COL)

validation_forecasts = benchmark_forecasts(train.to_numpy(dtype=float).tolist(), validation)
test_history = pd.concat([train, validation]).to_numpy(dtype=float).tolist()
test_forecasts = benchmark_forecasts(test_history, test)

validation_forecasts.head()


In [ ]:
def add_ses_forecast(forecasts: pd.DataFrame, fit_series: pd.Series) -> float:
    ses_model = SimpleExpSmoothing(
        fit_series.to_numpy(dtype=float),
        initialization_method="estimated",
    ).fit(optimized=True)

    alpha = float(ses_model.params["smoothing_level"])
    level = float(ses_model.level[-1])
    ses_values = []

    for actual in forecasts["actual"].to_numpy(dtype=float):
        ses_values.append(level)
        level = alpha * float(actual) + (1 - alpha) * level

    forecasts["ses"] = ses_values
    return alpha

validation_ses_alpha = add_ses_forecast(validation_forecasts, train)
test_ses_alpha = add_ses_forecast(test_forecasts, pd.concat([train, validation]))

print(f"Validation SES alpha: {validation_ses_alpha:.6f}")
print(f"Test SES alpha: {test_ses_alpha:.6f}")


## ARIMA Order Selection

ARIMA hyperparameters are selected on the training target only. This keeps validation and test data out of model selection.


In [ ]:
order_rows = []
train_values = train.to_numpy(dtype=float)

for d in [0, 1]:
    for p in range(4):
        for q in range(4):
            trend_options = ["n", "c"] if d == 0 else ["n"]
            for trend in trend_options:
                try:
                    model = ARIMA(
                        train_values,
                        order=(p, d, q),
                        trend=trend,
                        enforce_stationarity=False,
                        enforce_invertibility=False,
                    )
                    result = model.fit()
                    order_rows.append({
                        "p": p,
                        "d": d,
                        "q": q,
                        "trend": trend,
                        "aic": float(result.aic),
                        "bic": float(result.bic),
                    })
                except Exception:
                    pass

order_search = pd.DataFrame(order_rows).sort_values("aic").reset_index(drop=True)
order_search.to_csv(OUT_DIR / "arima_order_search.csv", index=False)
order_search.head(10)


In [ ]:
best_order = order_search.iloc[0]
best_p = int(best_order["p"])
best_d = int(best_order["d"])
best_q = int(best_order["q"])
best_trend = str(best_order["trend"])

arima_model = ARIMA(
    train.to_numpy(dtype=float),
    order=(best_p, best_d, best_q),
    trend=best_trend,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
arima_result = arima_model.fit()

print(f"Selected ARIMA({best_p}, {best_d}, {best_q}), trend={best_trend}")
print(f"AIC: {best_order['aic']:.6f}")
print(f"BIC: {best_order['bic']:.6f}")


## ARIMA Walk-Forward Forecasts

Validation uses a model fitted on the training segment. Test uses the same selected order, refitted on train + validation, then updated one observation at a time without refitting parameters.


In [ ]:
def arima_walk_forward(fit_series: pd.Series, actuals: pd.Series, order: tuple[int, int, int], trend: str) -> np.ndarray:
    state = ARIMA(
        fit_series.to_numpy(dtype=float),
        order=order,
        trend=trend,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit()

    predictions = []
    for actual in actuals.to_numpy(dtype=float):
        predictions.append(float(state.forecast(steps=1)[0]))
        state = state.append([float(actual)], refit=False)

    return np.asarray(predictions, dtype=float)

best_arima_order = (best_p, best_d, best_q)
validation_forecasts["arima"] = arima_walk_forward(train, validation, best_arima_order, best_trend)
test_forecasts["arima"] = arima_walk_forward(pd.concat([train, validation]), test, best_arima_order, best_trend)

test_forecasts.head()


## Residual Diagnostics

In [ ]:
arima_residuals = pd.Series(arima_result.resid, index=train.index).dropna()

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=False)

axes[0].plot(arima_residuals.index, arima_residuals, linewidth=0.8)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("ARIMA residuals over time")
axes[0].set_ylabel("Residual")

sns.histplot(arima_residuals, bins=60, kde=True, ax=axes[1], color="tab:blue")
axes[1].set_title("ARIMA residual distribution")
axes[1].set_xlabel("Residual")

plot_acf(arima_residuals, lags=40, ax=axes[2])
axes[2].set_title("ACF of ARIMA residuals")

plt.tight_layout()
plt.show()


In [ ]:
ljung_box_residuals = acorr_ljungbox(
    arima_residuals,
    lags=[5, 10, 20],
    return_df=True,
)
ljung_box_residuals.index.name = "lag"
ljung_box_residuals


## Benchmark Metrics

The main scale-dependent metrics are RMSE and MAE. Direction accuracy checks whether the model predicts the sign of the next daily return correctly.


In [ ]:
def model_metrics(actual: np.ndarray, prediction: np.ndarray) -> dict:
    actual = np.asarray(actual, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    error = prediction - actual

    denominator = np.abs(actual) + np.abs(prediction) + EPS
    sst = float(np.sum((actual - actual.mean()) ** 2))
    sse = float(np.sum(error ** 2))

    if np.std(prediction) > EPS and np.std(actual) > EPS:
        correlation = float(np.corrcoef(actual, prediction)[0, 1])
    else:
        correlation = np.nan

    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "mape_pct": float(np.mean(np.abs(error) / (np.abs(actual) + EPS)) * 100),
        "smape_pct": float(np.mean(2 * np.abs(error) / denominator) * 100),
        "r2": float(1 - sse / sst) if sst > EPS else np.nan,
        "correlation": correlation,
        "direction_accuracy": float(np.mean(np.sign(prediction) == np.sign(actual))),
    }


def evaluate_forecast_frame(forecasts: pd.DataFrame, split_name: str) -> pd.DataFrame:
    actual = forecasts["actual"].to_numpy(dtype=float)
    rows = []
    for model_name in ["naive", "rolling_7", "rolling_14", "rolling_30", "ses", "arima"]:
        rows.append({
            "split": split_name,
            "model": model_name,
            **model_metrics(actual, forecasts[model_name].to_numpy(dtype=float)),
        })
    return pd.DataFrame(rows)

validation_metrics = evaluate_forecast_frame(validation_forecasts, "validation")
test_metrics = evaluate_forecast_frame(test_forecasts, "test")
metrics = pd.concat([validation_metrics, test_metrics], ignore_index=True)
metrics.sort_values(["split", "rmse"])


In [ ]:
validation_forecasts.to_csv(OUT_DIR / "validation_predictions.csv")
test_forecasts.to_csv(OUT_DIR / "test_predictions.csv")
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)

metadata = {
    "target": TARGET_COL,
    "target_definition": "target_next_daily_log_return = daily_log_return.shift(-1)",
    "raw_5min_path": str(RAW_5MIN_PATH),
    "validation_start": val_start.isoformat(),
    "test_start": test_start.isoformat(),
    "selected_order": {"p": best_p, "d": best_d, "q": best_q},
    "trend": best_trend,
    "selection_metric": "aic",
    "best_aic": float(best_order["aic"]),
    "best_bic": float(best_order["bic"]),
    "train_rows": int(len(train)),
    "validation_rows": int(len(validation)),
    "test_rows": int(len(test)),
}

with (OUT_DIR / "arima_model_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(" -", OUT_DIR / "validation_predictions.csv")
print(" -", OUT_DIR / "test_predictions.csv")
print(" -", OUT_DIR / "metrics.csv")
print(" -", OUT_DIR / "arima_model_metadata.json")
metadata


## Forecast Plots

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(test_forecasts.index, test_forecasts["actual"], label="Actual", linewidth=1.2, color="black")
axes[0].plot(test_forecasts.index, test_forecasts["arima"], label="ARIMA", linewidth=1.0)
axes[0].plot(test_forecasts.index, test_forecasts["naive"], label="Naive", linewidth=0.9, alpha=0.8)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Actual vs forecasted next-day daily log return")
axes[0].set_ylabel("Log return")
axes[0].legend()

axes[1].plot(test_forecasts.index, test_forecasts["arima"] - test_forecasts["actual"], label="ARIMA error", linewidth=0.9)
axes[1].plot(test_forecasts.index, test_forecasts["naive"] - test_forecasts["actual"], label="Naive error", linewidth=0.9, alpha=0.8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Forecast errors")
axes[1].set_ylabel("Error")
axes[1].legend()

axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=5))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Final Summary

This ARIMA analysis is now aligned with the project target variable. It forecasts next-day daily log returns, while realized volatility remains relevant only as an explanatory diagnostic or as part of the hybrid model's volatility component.


In [ ]:
best_validation_model = validation_metrics.sort_values("rmse").iloc[0]
best_test_model = test_metrics.sort_values("rmse").iloc[0]

print(f"Best validation model by RMSE: {best_validation_model['model']}")
print(f"Validation RMSE: {best_validation_model['rmse']:.6f}")
print(f"Validation MAE: {best_validation_model['mae']:.6f}")
print(f"Validation direction accuracy: {best_validation_model['direction_accuracy']:.4f}")
print()
print(f"Best test model by RMSE: {best_test_model['model']}")
print(f"Test RMSE: {best_test_model['rmse']:.6f}")
print(f"Test MAE: {best_test_model['mae']:.6f}")
print(f"Test direction accuracy: {best_test_model['direction_accuracy']:.4f}")

metrics.sort_values(["split", "rmse"])
